In [1]:
TRAINING_FINISHED = False
RANDOM_STATE = 42
MAX_ITERATIONS = 5000
MAX_ITERATIONS_SVM = 1000000
TRAINING_SET_SIZE = 0.7
VALIDATION_TEST_SET_RATIO = 0.5

In [2]:
import pandas as pd

data = pd.read_csv('reviews.csv', sep=",",
                   usecols=['rating', 'review_text', 'helpful'])


In [3]:
data['rating'] = data['rating'].fillna(0)
data['helpful'] = data['helpful'].fillna(0)
data['review_text'] = data['review_text'].fillna("")

In [5]:
# lowercase all text
data['review_text'] = data['review_text'].str.lower()

In [6]:
import contractions

# expand contractions
data['review_text'] = data['review_text'].apply(contractions.fix)

In [7]:
import emoji

# interpret emojis as words
data['review_text'] = data['review_text'].apply(emoji.demojize)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
X_text = tfidf_vectorizer.fit_transform(data['review_text'])
X_helpful = csr_matrix(data['helpful'].values.reshape(-1, 1))

In [9]:
X = hstack([X_text, X_helpful])
y = data['rating']

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score

# Split base data into 2 sets, 30% for (test + validation), 70% train
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=1 - TRAINING_SET_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Split (test + validation) set into 50% (=15%) test and 50% (=15%) validation
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=VALIDATION_TEST_SET_RATIO, random_state=RANDOM_STATE, stratify=y_temp
)

In [11]:
models = []

In [12]:
from sklearn.svm import SVC

# Support Vector Machine model
svc_model = SVC(
    kernel='linear',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=MAX_ITERATIONS_SVM
)
svc_model.fit(X_train, y_train)
models.append(svc_model)

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [13]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
logreg_model = LogisticRegression(class_weight='balanced', max_iter=MAX_ITERATIONS)
logreg_model.fit(X_train, y_train)
models.append(logreg_model)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    max_depth=9,
    class_weight='balanced',
    n_estimators=300
)
rf_model.fit(X_train, y_train)
models.append(rf_model)

In [15]:
from sklearn.utils import compute_sample_weight
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

architectures = [
    (64,),
    (128, 64),
    (256, 128),
    (128, 64, 32),
    (128, 128, 64),
    (128, 128, 32),
]
best_score = 0
best_mlp = None

for i, arch in enumerate(architectures):
    print(f"Training architecture #{i} {arch}:")
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=MAX_ITERATIONS,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    mlp.fit(X_train, y_train, sample_weight=sample_weights)

    val_score = accuracy_score(y_validation, mlp.predict(X_validation))
    print(f"- Validation Accuracy: {val_score:.4f}")

    if val_score > best_score:
        best_score = val_score
        best_mlp = mlp

models.append(best_mlp)

Training architecture #0 (64,):
- Validation Accuracy: 0.5311
Training architecture #1 (128, 64):
- Validation Accuracy: 0.5451
Training architecture #2 (256, 128):
- Validation Accuracy: 0.5805
Training architecture #3 (128, 64, 32):
- Validation Accuracy: 0.4979
Training architecture #4 (128, 128, 64):
- Validation Accuracy: 0.5322
Training architecture #5 (128, 128, 32):
- Validation Accuracy: 0.4217


In [16]:
from sklearn.metrics import accuracy_score, classification_report

# PLOT THAT instead
for i, model in enumerate(models):
    print(f'Evaluate Model #{i}: {type(model).__name__}')
    # SVC uses scaled data
    y_pred = model.predict(X_validation)
    print(f'Accuracy: {accuracy_score(y_validation, y_pred):.4f}')
    print(classification_report(y_validation, y_pred))

    # print("Cross-validation scores: {}".format(cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')))
    # print("Confusion matrix:\n{}".format(confusion_matrix(y_validation, y_pred)))


Evaluate Model #0: SVC
Accuracy: 0.4839
              precision    recall  f1-score   support

           1       0.51      0.60      0.55       237
           2       0.13      0.17      0.15        70
           3       0.16      0.32      0.21        76
           4       0.08      0.06      0.07        95
           5       0.79      0.59      0.68       454

    accuracy                           0.48       932
   macro avg       0.33      0.35      0.33       932
weighted avg       0.55      0.48      0.50       932

Evaluate Model #1: LogisticRegression
Accuracy: 0.4785
              precision    recall  f1-score   support

           1       0.53      0.57      0.55       237
           2       0.14      0.19      0.16        70
           3       0.17      0.37      0.23        76
           4       0.11      0.11      0.11        95
           5       0.80      0.57      0.67       454

    accuracy                           0.48       932
   macro avg       0.35      0.36   

In [1]:
# FINAL SCORING; ONLY USE AFTER FINISHED TRAINING
if not TRAINING_FINISHED:
    print("Skipping final test — set TRAINING_FINISHED = True when ready.")
else:
    for i, model in enumerate(models):
        print(f'Evaluate Model #{i} with type: {type(model).__name__}')
        y_pred = model.predict(X_test)
        print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
        print(classification_report(y_test, y_pred))

NameError: name 'TRAINING_FINISHED' is not defined